# Pre-Training An LLM

💡 **Pre-training for the non-multi-modal architecture!**


🌟 **LLM training occurs in two phases:**

1. **Pre-Training** (Phase 1):
    - Objective: (Teacher Forcing) Next-Token prediction over a massive, unstructured dataset (trillions of words from the internet, books, code).
    - Note: The pre-training phase is the most expensive part of training an LLM.
    - Goal: To teach the model the structure of human language, logic, coding syntax, and general world knowledge. This is creates `base` model.
    - Steps:
        1. Pass a input sequence of tokens through the model. Multiple documents are packed into a single sequence.
            - The tokens are masked so the model can not see future tokens, and specific to Llama, a document mask that prevents a token from one document from attending to tokens in other documents in the same sequence.
            - The ground truth are the inputs shifted by one token to the left.
        2. The models outputs logits for every position simultaneously, and we calculate cross-entropy loss between the model's prediction and the ground truth.

2. [**Post-Training** (Phase 2)](./post_training.ipynb)

---

### Llama 3 paper

> **Language model pre-training.**
>
> "We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train a model with $405B$ parameters on $15.6T$ tokens using a context window of $8K$ tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to $128K$ tokens. See Section 3 for details."
>
> **3.4.1 Initial Pre-Training**
>
> We pre-train Llama $3$ $405\text{B}$ using AdamW with a peak learning rate of $8 × 10^{−5}$ , a linear warm up of $8{,}000$ steps, and a cosine learning rate schedule decaying to $8 × 10^{−7}$ over $1{,}200{,}000$ steps. We use a lower batch size early in training to improve training stability, and increase it subsequently to improve efficiency. Specifically, we use an initial batch size of $4\text{M}$ tokens and sequences of length $4{,}096$, and double these values to a batch size of $8\text{M}$ sequences of $8{,}192$ tokens after pre-training $252\text{M}$ tokens. We double the batch size again to 16M after pre-training on $2.87\text{T}$ tokens. We found this training recipe to be very stable: we observed few loss spikes and did not require interventions to correct for model training divergence.
>
> **3.4.2 Long Context Pre-Training**
>
> In the final stages of pre-training, we train on long sequences to support context windows of up to $128\text{K}$ tokens. We do not train on long sequences earlier because the compute in self-attention layers grows quadratically in the sequence length. We increase the supported context length in increments, pre-training until the model has successfully adapted to the increased context length. We assess successful adaptation by measuring whether (1) model performance on short-context evaluations has recovered completely and (2) the model perfectly solves “needle in a haystack” tasks up to that length. In Llama $3$ $405\text{B}$ pre-training, we increased context length gradually in six stages, starting from the original $8\text{K}$ context window and ending in the final $128\text{K}$ context window. This long-context pre-training stage was performed using approximately $800\text{B}$ training tokens.
>
> **3.4.3 Annealing**
>
> During pre-training on the final $40\text{M}$ tokens, we linearly annealed the learning rate to $0$, maintaining a context length of $128\text{K}$ tokens. During this annealing phase, we also adjusted the data mix to upsample data sources of very high quality; see Section 3.1.3. Finally, we compute the average of model checkpoints (Polyak (1991) averaging) during annealing to produce the final pre-trained model.

---


**Paper Deconstruct:**

- This was the recipe to pretrain teh **Llama 3 405B**, but all models follow the same pipeline with different values.
- Pre-training consists of stages:
    1. **Initial (~15.6T tokens)**:
        - NOTE: Sequence length = Context len/window size.
        - It was trained for $1{,}200{,}000$ steps on $15.6\text{T}$ token corpus.
        - Warmup ($8{,}000$ steps): Warms-up over the first $8{,}000$ steps to reach a peak learning rate = $8 × 10^{−5}$
        - Then it decays the learning rate back down to $8 × 10^{−7}$ over $1{,}200{,}000$ steps.
        - The `batch_size` starts at $4\text{M}$ tokens and the `seq_len` starts at $4{,}096$
            - So $\text{num sequences} = \frac{4\text{M} tokens}{4{,}096} \approx 976$
            - Meaning $\approx 976$ sequences are processed per optimizer step in parallel across all GPUs.
        - After training on the first $252\text{M}$ tokens, the `batch_size` and the `seq_len` are doubled to $8\text{M}$ and $8{,}192$ respectively.
        - After training on $2.87\text{T}$ tokens, the `batch_size` is doubled again to $16\text{M}$

    2. **Long-Context (~800B tokens):**
        - Starts after the initial phase is complete.
        - LR is kept at `decay_lr_to` = $8 × 10^{−7}$
        - The dataset must contain long sequences to support the $128\text{K}$ tokens.
        - Context length/`seq_len` is increased gradually **across 6 stages** from $8{,}192$ tokens to $128\text{K}$ tokens.
            - This is done because the model has learned how to predict words, it just needed to learn how to look further back in its memory to do it.
        - Since the context length is higher, it means that the rate of tokens consumed per step is much higher in this stage than the pervious stage.
    3. **Annealing (Final 40M tokens):**
        - Learning rate is linearly decayed to $0$ over the last 40M tokens, the seq_len is maintained at $128\text{K}$
        - The dataset is upsampled to the absolute highest quality data mix.
        - Finally, checkpoints are averaged via Polyak averaging.


⭐️ [Optimizer and Learning Rate](../optimizer_and_lr_schedule.ipynb)


---

## Validation

I used HuggingFace's [Lighteval](https://huggingface.co/docs/lighteval/en/index#lighteval)

Here is the [guide](https://huggingface.co/docs/lighteval/en/evaluating-a-custom-model) for using it on a custom model.

[More info in run_evaluate.py](../../run_evaluate.py)

### Llama 3 paper

> **3.1.3 Annealing Data**
>
> Empirically, we find that annealing (see Section 3.4.3) on small amounts of high-quality code and mathematical
data can boost the performance of pre-trained models on key benchmarks. Akin to Li et al. (2024b), we
perform annealing with a data mix that upsamples high-quality data in select domains. We do not include
any training sets from commonly used benchmarks in our annealing data. This enables us to assess the true
few-shot learning capabilities and out-of-domain generalization of Llama 3.
>
> Following OpenAI (2023a), we evaluate the efficacy of annealing on the **GSM8k** (Cobbe et al., 2021) and
**MATH** (Hendrycks et al., 2021b) training sets in annealing. We find that annealing improved the performance
of a **pre-trained Llama 3 8B model on the GSM8k and MATH validation sets** by 24.0% and 6.4%, respectively.
However, the improvements on the 405B model are negligible, **suggesting that our flagship model has strong
in-context learning and reasoning capabilities and does not require specific in-domain training samples to
obtain strong performance.**
>
> Using annealing to assess data quality. Similar to Blakeney et al. (2024), we find that annealing enables us to
judge the value of small domain-specific datasets. We measure the value of such datasets by annealing the
learning rate of a 50% trained Llama 3 8B model linearly to 0 on 40B tokens. In those experiments, we assign
30% weight to the new dataset and the remaining 70% weight to the default data mix. Using annealing to
evaluate new data sources is more efficient than performing scaling law experiments for every small dataset.
>
> **5.2.6 Long Context Benchmarks**
> 
> We consider a diverse set of tasks that span various domains and text types. In the benchmarks we list below,
we focus on sub-tasks that use unbiased evaluation protocols, i.e., accuracy-based metrics rather than n-gram
overlapping metrics. We also prioritize tasks that we found to be of low variance.
>
> - **Needle-in-a-Haystack** (Kamradt, 2023) measures a model’s ability to retrieve a hidden information
inserted in random parts of the long document. Our Llama 3 models demonstrate perfect needle retrieval
performance, successfully retrieving 100% of needles at all document depths and context lengths. We
also measure performance on Multi-needle (Table 21), a variation of Needle-in-a-Haystack, where we
insert four needles in the context and test if a model can retrieve two of them. Our Llama 3 models
achieve near perfect retrieval results.
> - **ZeroSCROLLS** (Shaham et al., 2023) is a zero-shot benchmark for natural language understanding over
long texts. We report numbers on the validation set, as the ground truth answers are not publicly
available. Our Llama 3 405B and 70B models either match or surpass other models on various tasks in
this benchmark.
> - **InfiniteBench** (Zhang et al., 2024) requires models to understand long dependencies in the context
window. We evaluate Llama 3 on En.QA (QA over novels) and En.MC (multiple-choice QA over novels),
where our 405B model outperforms all others. The gains are particularly significant on En.QA.

---

**Paper Deconstruct:**

- 🚨 **Note:** The paper mentions very little about the actual validation pipeline other than the benchmarks. From what I could find, a typical LLM validation works like so:

    1. During the pre-training loop, after a certain number of steps or tokens, we validate on data the model has not seen yet to detect: Divergence, overfitting, and check other diagnostics. This is often called the **"online"** validation.
    2. Benchmark Evaluation: When a checkpoint is saved during the training loop, we run a separate evaluation script on that checkpoint using a benchmark (MATH, GSM8k, etc... depending on which stage that checkpoint was in). This is often called the **"offline"** validation. Checkout [pre-training_eval.ipynb](./pre-training_eval.ipynb).

- A different set of benchmarks is used per stage to validate that stage's specific goal. 
  1. Initial Stage:
     - Benchmarks: Paper did not specify.
  2. Long-Context Stage:
     - Benchmarks: Needle-in-a-Haystack, ZeroSCROLLS, InfiniteBench. 
     - Reasoning: As we increase `seq_len`, we need to ensure the model is capable of finding information hidden in massive documents.
  3. Annealing Stage:
     - Benchmarks: GSM8k and MATH. 
     - Reasoning: We use the highest quality dataset during this stage, while decaying the learning rate to zero, we need to ensure the model has a reasonable reasoning score on these benchmarks.

### Why "Online" Vs "Offline" Validations?

The **online validation** in `PreTrainTextOnly` training loop tracks standard cross-entropy loss to catch catastrophic divergence early, to minimize wasted compute. 

The **offline validation** (benchmark evaluation) is what determines if the model is learning useful skills like reasoning, retrieval, etc...

#### How To Run Offline Validation

**Once a checkpoint has been saved (during the long-context or annealing stage!). Run:**


```bash
# cd into project root
# Optional flags:
#    --overfit   but note results will be poor!
python run_evaluate.py --chpt <path_to_the_checkpoint_dir_that_contains_checkpoint.pt>
```

## Loss Function

**Llama 3 Paper:**

> "Language model **pre-training**. **We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction.** **In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”.** To do this effectively, pre-training is performed at massive scale: we pre-train a model with 405B parameters on 15.6T tokens using a context window of 8K tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to 128K tokens. See Section 3 for details."


## Pre-Training


In [1]:
import easyjupyter
import torch
import torch.nn as nn
from pathlib import Path
import time
from datetime import datetime, timedelta
from model.utils.model_io import save_checkpoint
from model.utils.plot import plot_loss_history
from model.utils.masking import create_causal_doc_mask
from typing import TYPE_CHECKING, List
from model.data_loader import create_pretrain_dataloader
from model.utils.check_sys_vram import get_max_micro_batch_size
from model.gradient_accumulation import GradientAccumulation
from model.utils.chpt_averaging import average_checkpoints

/Users/tonyavis/miniconda3/envs/build_an_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from configs import ConfigTemplate
    import torch.optim as optim
    from model.model_text_only import TextOnlyModel

In [3]:
class PreTrainTextOnly:
    def __init__(
        self,
        cfg: ConfigTemplate,
        model: TextOnlyModel,
        optimizer: optim.Optimizer,
        is_overfit: bool = False,
    ):
        """Implements pre-training."""
        self.cfg = cfg
        self.model = model
        self.is_overfit = is_overfit

        self.criterion = nn.CrossEntropyLoss(
            ignore_index=cfg.special_tokens["pad_token"]["ID"],
        )
        self.optimizer = optimizer
        self.accum_tracker = GradientAccumulation()

        self.loss_history = []
        self.val_loss_history = []
        self.lr_history = []
        self.diagnostic_history = []
        self.total_tokens_processed = 0  # Track the total tokens for all stages.
        self.chpts_paths: list[Path] = []

        # Exponential Moving Average for prints
        self.start_time = None
        self.alpha = 0.1
        self.eta_time = None
        self.last_print_time = None
        self.initial_stage_start_time = None
        self.lc_stage_start_time = None
        self.annealing_stage_start_time = None

        # Print String Formatting
        self.highestTokenCount = cfg.text_only.pretrain.initial_stage.tokens
        self.width = len((f"{self.highestTokenCount:,}"))
        self.highestStepWidth = len(str(cfg.text_only.pretrain.initial_stage.steps))
        self.step_block_width = len(
            f"Step: {cfg.text_only.pretrain.initial_stage.steps}/{cfg.text_only.pretrain.initial_stage.steps}"
        )
        self.fmt = {"avg_time": 5, "elapsed": 8, "eta": 8, "lr": 15, "loss": 7}

    def train(self, print_every: int = 100):
        print("\n" + "#" * 64)
        print(
            f"\nPre-Training Model | Active Stage: {self.cfg.text_only.pretrain.curr_stage_name} "
        )
        print("\n" + "#" * 64)

        self.model.train()  # Set the model to train mode
        self.start_time = time.time()
        self.last_print_time = self.start_time

        data_iter = None
        x_ids, y_ids = None, None
        running_loss = 0.0  # track running loss to accurately sum the scaled losses over the micro-batches

        self.optimizer.zero_grad()  # Clear gradients before starting

        # Use a single sample for the overfit test
        overfit_x = None
        overfit_y = None

        while self.cfg.text_only.pretrain.curr_stage_name != "complete":
            stage_name = self.cfg.text_only.pretrain.curr_stage_name
            stage_cfg = getattr(self.cfg.text_only.pretrain, stage_name)

            # --- Check Stage Completion ---
            if stage_cfg.is_complete():

                micro_steps_taken = (
                    self.accum_tracker.current_micro_step
                    % self.accum_tracker.accumulation_steps
                )
                # --- Flush stranded gradients that weren't accumulated before transitioning to the next stage
                # Example: accumulation_steps = 4, but we only made two micro_steps, then we are left with those 2 micro-steps, and they are stranded between stages. So, we need to accurately accumulate them.
                if micro_steps_taken != 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), max_norm=1.0
                    )
                    self.optimizer.step()
                    self.optimizer.zero_grad()

                    # Because this was a partial accumulation, the running_loss must be scaled up to represent the true average
                    # of just these partial micro-steps.
                    adjusted_loss = running_loss * (
                        self.accum_tracker.accumulation_steps / micro_steps_taken
                    )
                    self.lr_history.append(self.optimizer.param_groups[0]["lr"])
                    self.loss_history.append(adjusted_loss)
                    self.diagnostic_history.append(
                        {
                            "seq_len": self.cfg.curr_seq_len,
                            "micro_batch": self.cfg.curr_micro_batch_size,
                            "global_batch_tokens": self.cfg.curr_global_batch_tokens,
                            "total_tokens_processed": self.total_tokens_processed,
                        }
                    )

                    self.cfg.num_grad_updates += 1
                    self._print_step_info(adjusted_loss, print_every)
                    running_loss = 0.0

                self._transition_to_next_stage()
                if self.cfg.text_only.pretrain.curr_stage_name == "complete":
                    break

                # Unset data_iter for force the shape update block to fetch the new stage's dataloader
                data_iter = None
                continue

            # --- Check Shape Milestones ---
            expected_seq_len, expected_global_tokens = stage_cfg.get_target_shapes(
                self.cfg.curr_seq_len,
                self.cfg.curr_global_batch_tokens,
            )

            if (
                expected_seq_len != self.cfg.curr_seq_len
                or expected_global_tokens != self.cfg.curr_global_batch_tokens
                or self.cfg.curr_micro_batch_size == 0
                or data_iter is None
            ):
                # One of the stages is updating a shape, either seq_len, global_batch_tokens
                print(
                    f"\n[SHAPE UPDATE] Target seq_len: {expected_seq_len} | Target global_tokens: {expected_global_tokens}"
                )

                if self.is_overfit:
                    # Bypass hardware profiting to maintain a tiny footprint for the overfit test
                    safe_micro_batch = 2
                    print(f"[OVERFIT] Forcing micro_batch_size: {safe_micro_batch}")
                else:
                    # Get the max micro_batch_-size for this specific seq_len that can fit into the current system's VRAM.
                    max_hardware_micro_batch = get_max_micro_batch_size(
                        self.cfg,
                        self.model,
                        expected_seq_len,
                    )

                    # Leave at least 15% memory buffer just in case of optimization state spikes
                    safe_micro_batch = max(1, int(max_hardware_micro_batch * 0.85))
                    print(
                        f"[VRAM PROFILER] Assigned micro_batch_size: {safe_micro_batch}"
                    )

                self.cfg.curr_micro_batch_size = safe_micro_batch
                self.cfg.curr_seq_len = expected_seq_len
                self.cfg.curr_global_batch_tokens = expected_global_tokens

                # Update the accumulation targets on the fly
                accum_steps = self.accum_tracker.update_shape_target(
                    global_tokens=expected_global_tokens,
                    micro_batch_size=safe_micro_batch,
                    seq_len=expected_seq_len,
                )
                print(
                    f"[ACCUMULATION] Hardware VRAM capacity is {safe_micro_batch * expected_seq_len} tokens/pass. Setting accumulation steps to {accum_steps}."
                )

                if not self.is_overfit:
                    # Rebuild dataloader for new seq_len and batch_size_tokens shapes
                    if data_iter is not None:
                        del data_iter
                    dataloader = self._get_current_dataloader()
                    data_iter = iter(dataloader)
                else:
                    # Overfit mode: use a single batch for all
                    if overfit_x is None:
                        dataloader = self._get_current_dataloader()
                        overfit_x, overfit_y = next(iter(dataloader))
                        overfit_x = overfit_x.to(self.cfg.device)
                        overfit_y = overfit_y.to(self.cfg.device)
                    data_iter = True  # assign dummy value so `data_iter` stops the infinite loop

            # --- Fetch Data ---
            if not self.is_overfit:
                x_ids, y_ids = next(data_iter)
                x_ids, y_ids = x_ids.to(self.cfg.device), y_ids.to(self.cfg.device)
            else:
                x_ids, y_ids = self.overfit_data(overfit_x, overfit_y)

            # --- Forward & Backward Pass ---
            # Generate the doc and causal mask
            batch_mask = create_causal_doc_mask(cfg=self.cfg, token_ids=x_ids)

            # NOTE: My M1 Mac does not support bfloat16, but later M3 onward support bfloat16
            autocast_dtype = (
                torch.float16 if self.cfg.device.type == "mps" else torch.bfloat16
            )

            # Use autocast() to both save memory, and speed up training.
            with torch.autocast(device_type=self.cfg.device.type, dtype=autocast_dtype):
                # Forward pass
                logits, _ = self.model(x_ids, batch_mask)

                # Compute the loss & Flatten to 2D and 1D for CrossEntropy
                loss = self.criterion(
                    logits.reshape(-1, self.cfg.vocab_size), y_ids.reshape(-1)
                )

                # Scale the accumulated gradients by the accumulation steps to match it as if it was a single global batch.
                loss = self.accum_tracker.scale_loss(loss)

            # Backward pass
            loss.backward()
            running_loss += loss.item()

            # --- Update the stage config ---
            tokens_in_batch = x_ids.numel()
            stage_cfg.tokens_processed += tokens_in_batch
            self.total_tokens_processed += tokens_in_batch
            self.accum_tracker.current_micro_step += 1

            # --- Optimizer Step only triggers every N micro-steps
            if self.accum_tracker.should_step:
                # Clip gradients to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                # Step the optimizer
                self.optimizer.step()
                self.optimizer.zero_grad()  # Clear gradients for next iteration

                # Update LR
                lr = stage_cfg.get_learning_rate(
                    initial_min_lr=self.cfg.text_only.pretrain.initial_stage.decay_lr_to
                )
                self.lr_history.append(lr)

                # Inject learning rate straight into the optimizer parameter groups
                for param_group in self.optimizer.param_groups:
                    param_group["lr"] = lr

                # Update global trackers
                if stage_name == "initial_stage":
                    stage_cfg.steps_completed += 1
                self.cfg.num_grad_updates += 1

                self.loss_history.append(running_loss)

                self.diagnostic_history.append(
                    {
                        "seq_len": self.cfg.curr_seq_len,
                        "micro_batch": self.cfg.curr_micro_batch_size,
                        "global_batch_tokens": self.cfg.curr_global_batch_tokens,
                        "total_tokens_processed": self.total_tokens_processed,
                    }
                )

                self._print_step_info(
                    loss=running_loss,
                    print_every=print_every,
                )
                running_loss = 0.0

                if self.cfg.num_grad_updates % 200 == 0 or (
                    self.is_overfit and self.cfg.num_grad_updates % 50 == 0
                ):
                    self._run_validation()
                    self._save_checkpoint()

        self._save_checkpoint()

        # === Polyak Checkpoint Averaging ===
        if self.chpts_paths:
            final_chpt_dir = self.cfg.get_chpt_save_path(
                self.cfg.text_only.pretrain.save_dir_name,
                optionalPrefix="final_averaged_",
            )
            average_checkpoints(self.chpts_paths, final_chpt_dir)
            self.cfg.save(final_chpt_dir)

        else:
            print(
                "[WARNING] No annealing-stage checkpoints collected; skipping averaging, and you can not use any of the checkpoints for post-training yet!"
            )

        print("\n>>> Pre-Training complete! <<<\n")

    def _get_current_dataloader(self):
        curr_stage_name = self.cfg.text_only.pretrain.curr_stage_name
        bin_path = self.cfg.data_dir / f"{self.cfg.prefix}{curr_stage_name}.bin"

        if curr_stage_name == "initial_stage":
            # During the initial stage the batch_size and seq_len are increased
            stage_cfg = self.cfg.text_only.pretrain.initial_stage

            self.cfg.curr_seq_len, self.cfg.curr_global_batch_tokens = (
                stage_cfg.get_target_shapes(
                    self.cfg.curr_seq_len, self.cfg.curr_global_batch_tokens
                )
            )
            self.initial_stage_start_time = time.time()

        elif curr_stage_name == "lc_stage":
            # During the long-context stage the seq_len is increased gradually in 6 stages
            stage_cfg = self.cfg.text_only.pretrain.lc_stage
            self.cfg.curr_seq_len, self.cfg.curr_global_batch_tokens = (
                stage_cfg.get_target_shapes(
                    self.cfg.curr_seq_len, self.cfg.curr_global_batch_tokens
                )
            )
            self.lc_stage_start_time = time.time()

        elif curr_stage_name == "annealing_stage":
            stage_cfg = self.cfg.text_only.pretrain.annealing_stage
            self.cfg.curr_seq_len, self.cfg.curr_global_batch_tokens = (
                stage_cfg.get_target_shapes(
                    self.cfg.curr_seq_len, self.cfg.curr_global_batch_tokens
                )
            )
            self.annealing_stage_start_time = time.time()

        else:
            raise ValueError(f"Invalid stage name: {curr_stage_name}")

        tokens_seen = getattr(
            self.cfg.text_only.pretrain, curr_stage_name
        ).tokens_processed

        return create_pretrain_dataloader(
            cfg=self.cfg,
            bin_path=bin_path,
            tokens_seen=tokens_seen,
        )

    def _transition_to_next_stage(self):
        old_stage_name = self.cfg.text_only.pretrain.curr_stage_name

        if old_stage_name == "initial_stage":
            self.cfg.text_only.pretrain.curr_stage_name = "lc_stage"
        elif old_stage_name == "lc_stage":
            self.cfg.text_only.pretrain.curr_stage_name = "annealing_stage"
        else:
            self.cfg.text_only.pretrain.curr_stage_name = "complete"

        if self.cfg.text_only.pretrain.curr_stage_name != "complete":
            print(
                f"\n\n[STAGE TRANSITION] MOVING into stage: {self.cfg.text_only.pretrain.curr_stage_name}"
            )

        # Record the exact num_grad_updates when we enter a new stage
        self.cfg.text_only.pretrain.transition_steps[
            self.cfg.text_only.pretrain.curr_stage_name
        ] = self.cfg.num_grad_updates

    def _print_step_info(self, loss, print_every):
        curr_time = time.time()

        internal_time = curr_time - self.last_print_time
        actual_time = internal_time / print_every

        if self.eta_time is None:
            self.eta_time = actual_time
        else:
            self.eta_time = (self.alpha * actual_time) + (
                1 - self.alpha
            ) * self.eta_time

        stage_name = self.cfg.text_only.pretrain.curr_stage_name
        stage_cfg = getattr(self.cfg.text_only.pretrain, stage_name)

        tokens_display = (
            f"{stage_cfg.tokens_processed:0{self.width},d}/{stage_cfg.tokens:,}"
        )

        if stage_name == "initial_stage":
            if stage_cfg.tokens_processed == 0:
                eta_seconds = 0
            else:
                elapsed = curr_time - self.initial_stage_start_time
                steps_per_sec = stage_cfg.steps_completed / elapsed
                remaining_steps = stage_cfg.steps - stage_cfg.steps_completed
                eta_seconds = remaining_steps / steps_per_sec

            step_str = f"Step: {stage_cfg.steps_completed:>{self.highestStepWidth}d}/{stage_cfg.steps:>{self.highestStepWidth}d}"
            stage_str = f"{step_str} | Tokens: {tokens_display}"

        elif stage_name == "lc_stage":
            if self.lc_stage_start_time is None or stage_cfg.tokens_processed == 0:
                eta_seconds = 0
            else:
                elapsed = curr_time - self.lc_stage_start_time
                tokens_per_sec = stage_cfg.tokens_processed / elapsed
                remaining_tokens = stage_cfg.tokens - stage_cfg.tokens_processed
                eta_seconds = remaining_tokens / tokens_per_sec

            # Pad with exact spaces to match initial stage print
            stage_str = f"{'':>{self.step_block_width}} | Tokens: {tokens_display}"

        elif stage_name == "annealing_stage":
            if (
                self.annealing_stage_start_time is None
                or stage_cfg.tokens_processed == 0
            ):
                eta_seconds = 0
            else:
                elapsed = curr_time - self.annealing_stage_start_time
                tokens_per_sec = stage_cfg.tokens_processed / elapsed
                remaining_tokens = stage_cfg.tokens - stage_cfg.tokens_processed
                remaining_tokens = stage_cfg.tokens - stage_cfg.tokens_processed
                eta_seconds = remaining_tokens / tokens_per_sec

            # Pad with exact spaces to match initial stage print
            stage_str = f"{'':>{self.step_block_width}} | Tokens: {tokens_display}"
        else:
            raise ValueError(f"Unknown stage name: {stage_name}")

        eta_str = str(timedelta(seconds=int(max(eta_seconds, 0))))
        elapsed_str = str(timedelta(seconds=int(curr_time - self.start_time)))

        print(
            f"[{datetime.now().strftime('%m-%d %H:%M:%S')}] | "
            f"num_grad_updates: {self.cfg.num_grad_updates:{self.highestStepWidth}d} | "
            f"{stage_str} | "
            f"Avg Time: {self.eta_time:>{self.fmt['avg_time']}.2f}s | "
            f"Elapsed: {elapsed_str:>{self.fmt['elapsed']}} | "
            f"ETA: {eta_str:>{self.fmt['eta']}} | "
            f"LR: {self.lr_history[-1]:>{self.fmt['lr']}.8f} | "
            f"Loss: {loss:>{self.fmt['loss']}.4f}"
        )
        self.last_print_time = curr_time

    def _save_checkpoint(self):
        save_path = self.cfg.get_chpt_save_path(
            self.cfg.text_only.pretrain.save_dir_name
        )

        save_checkpoint(
            self.cfg,
            self.model,
            self.optimizer,
            self.loss_history,
            save_path,
        )

        if (
            self.cfg.text_only.pretrain.curr_stage_name == "annealing_stage"
            or self.cfg.text_only.pretrain.curr_stage_name == "complete"
        ):
            self.chpts_paths.append(save_path)

        plot_loss_history(
            self.cfg,
            self.loss_history,
            self.lr_history,
            save_path,
            val_loss_history=self.val_loss_history,
            diagnostic_history=self.diagnostic_history,
        )

    def _run_validation(self):
        """
        Run "online" validation to detect: Divergence, overfitting, and check other diagnostics.
        """
        self.model.eval()
        curr_stage_name = self.cfg.text_only.pretrain.curr_stage_name
        val_bin_path = self.cfg.data_dir / f"{self.cfg.prefix}{curr_stage_name}_val.bin"

        # Create a temp dataloader for the validation binary
        val_dataloader = create_pretrain_dataloader(
            cfg=self.cfg,
            bin_path=val_bin_path,
            tokens_seen=0,  # Always start validation from the beginning
            is_val=True,
        )

        total_val_loss = 0.0
        val_steps = 0

        autocast_dtype = (
            torch.float16 if self.cfg.device.type == "mps" else torch.bfloat16
        )

        print(f"\n--- Running Validation for {curr_stage_name} ---")

        with torch.no_grad():
            with torch.autocast(device_type=self.cfg.device.type, dtype=autocast_dtype):
                for x_val, y_val in val_dataloader:
                    x_val, y_val = x_val.to(self.cfg.device), y_val.to(self.cfg.device)
                    batch_mask = create_causal_doc_mask(cfg=self.cfg, token_ids=x_val)
                    logits, _ = self.model(x_val, batch_mask)
                    loss = self.criterion(
                        logits.reshape(-1, self.cfg.vocab_size), y_val.reshape(-1)
                    )

                    total_val_loss += loss.item()
                    val_steps += 1

        avg_val_loss = total_val_loss / max(1, val_steps)
        self.val_loss_history.append((self.cfg.num_grad_updates, avg_val_loss))
        print(f"[VALIDATION complete] Avg Loss: {avg_val_loss:.4f}\n")

        self.model.train()

    def overfit_data(self, overfit_x, overfit_y):
        # Overfit: Dynamically tile and expand the single sample sequence to match the current shapes (seq_len, and batch_tokens)
        flat_x = overfit_x.flatten()
        flat_y = overfit_y.flatten()

        target_tokens = self.cfg.curr_seq_len
        if flat_x.size(0) < target_tokens:
            repeats = target_tokens // flat_x.size(0) + 1
            flat_x = flat_x.repeat(repeats)
            flat_y = flat_y.repeat(repeats)

        # Slice down to exact curr_seq_len
        seq_x = flat_x[: self.cfg.curr_seq_len]
        seq_y = flat_y[: self.cfg.curr_seq_len]
        # Expand to fill the active micro_batch_size
        x_ids = seq_x.unsqueeze(0).expand(self.cfg.curr_micro_batch_size, -1)
        y_ids = seq_y.unsqueeze(0).expand(self.cfg.curr_micro_batch_size, -1)
        return x_ids, y_ids

## OVERFIT TEST

`Run this to create the binary dataset`

```bash
python prepare.py --stage pretrain --overfit
```


In [4]:
# @i-c
print("\n ---------- OVERFIT TEST ---------- \n")
from configs import Scaled_down_text
from model.model_text_only import TextOnlyModel
from model.optimizer_and_lr_schedule import get_optimizer

Scaled_down_text.initialize(is_overfit=True)
stage_bins = [
    "overfit_initial_stage.bin",
    "overfit_lc_stage.bin",
    "overfit_annealing_stage.bin",
]

if not all((Scaled_down_text.data_dir / f).exists() for f in stage_bins):
    raise ValueError(
        f"Binary datasets does not exist.\n"
        "Please run `python prepare.py --stage pretrain --overfit` first."
    )


model = TextOnlyModel(Scaled_down_text).to(Scaled_down_text.device)
optimizer = get_optimizer(Scaled_down_text, model=model)

trainer = PreTrainTextOnly(Scaled_down_text, model, optimizer, is_overfit=True)


 ---------- OVERFIT TEST ---------- 


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: scaled_down_text
Configured for overfitting...


Model initialized with 16,972,928 parameters!




Took about **~12 seconds** to run the overfit test on my M1 Mac Max.


In [5]:
# @i-c:
print(
    "If the training loss approaches 0.0, the model architecture and gradient flow are functional.\n"
    "NOTE: Because this is an overfit test, the validation loss result matters very little here! It is only for testing purposes.\n"
)
trainer.train(print_every=25)

If the training loss approaches 0.0, the model architecture and gradient flow are functional.
NOTE: Because this is an overfit test, the validation loss result matters very little here! It is only for testing purposes.


################################################################

Pre-Training Model | Active Stage: initial_stage 

################################################################

[SHAPE UPDATE] Target seq_len: 128 | Target global_tokens: 256
[OVERFIT] Forcing micro_batch_size: 2
[ACCUMULATION] Hardware VRAM capacity is 256 tokens/pass. Setting accumulation steps to 1.
[06-24 16:06:37] | num_grad_updates:   1 | Step:   1/100 | Tokens: 00,256/20,000 | Avg Time:  0.02s | Elapsed:  0:00:00 | ETA:  0:00:55 | LR:      0.00000000 | Loss: 11.0622
[06-24 16:06:37] | num_grad_updates:   2 | Step:   2/100 | Tokens: 00,512/20,000 | Avg Time:  0.02s | Elapsed:  0:00:00 | ETA:  0:00:29 | LR:      0.00010000 | Loss: 10.9338
[06-24 16:06:37] | num_grad_updates:   3 | Step:   3/100

![Training plot](../../showcase_images/overfit_training_loss_plot.png)